# Contrastive Dense Fine-Tuning

This pipeline mathematically bridges the gap between Masked Language Modeling (MLM) and Semantic Similarity.
The grammar-adapted `san_diego_legal_bert` model is explicitly trained using **Contrastive Loss (MultipleNegativesRankingLoss)** on the Ground Truth Question-Answer pairs so it learns to group similar sentences together in dense vector space.

In [1]:
# Install if missing
# !pip install sentence-transformers pandas datasets

import pandas as pd
import json
import os
from datasets import Dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# Paths
base_path = '..'
ground_truth_path = f"{base_path}/data/processed/evaluation_ground_truth.json"
mlm_model_path = f"{base_path}/models/san_diego_legal_bert"
output_model_path = f"{base_path}/models/san_diego_legal_bert_contrastive"

/Users/Aresh/Desktop/AAI 590/sd-land-use-rag/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/Aresh/Desktop/AAI 590/sd-land-use-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the Evaluation Dataset as Training Examples
The pairs of `(Question, Exact Ground Truth Chunk)` are extracted to act as Anchor/Positive training examples.

In [2]:
with open(ground_truth_path, 'r') as f:
    evaluation_data = json.load(f)

train_examples = []
for item in evaluation_data:
    # Provide the sentence-transformers InputExample object with texts=[Anchor, Positive]
    train_examples.append(InputExample(texts=[item['query'], item['source_text']]))

print(f"Loaded {len(train_examples)} Question-Answer pairs for contrastive training.")

# Load them into a Torch DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

Loaded 50 Question-Answer pairs for contrastive training.


## 2. Load the MLM Model & Define the Contrastive Loss
Using `MultipleNegativesRankingLoss`, the model is actively forced to pull the Ground Truth text closer to the query while pushing all other chunks in the batch away.

In [3]:
print("Loading MLM-adapted Legal-BERT...")
# Load our custom model from Stage 07. 
# SentenceTransformer natively handles wrapping bare BERT models into a pooled embedding architecture.
model = SentenceTransformer(mlm_model_path)

print("Initializing MultipleNegativesRankingLoss...")
train_loss = losses.MultipleNegativesRankingLoss(model=model)

No sentence-transformers model found with name ../models/san_diego_legal_bert. Creating a new one with mean pooling.
Some weights of BertModel were not initialized from the model checkpoint at ../models/san_diego_legal_bert and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading MLM-adapted Legal-BERT...
Initializing MultipleNegativesRankingLoss...


## 3. Fit the Model
The fine-tuning loop is run for 10 epochs. Since the dataset is small (50 pairs), this runs in seconds on local CPU.

In [4]:
print("Starting fine-tuning loop...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=10,
    warmup_steps=5,     # 10% of training data as warmup
    show_progress_bar=True
)

print("\nFine-tuning complete.")

Starting fine-tuning loop...


/Users/Aresh/Desktop/AAI 590/sd-land-use-rag/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss



Fine-tuning complete.


## 4. Save the Contrastive Model
The new weights are exported back into the `models/` directory for the final Benchmark run.

In [5]:
os.makedirs(output_model_path, exist_ok=True)
model.save(output_model_path)
print(f"Contrastive-adapted Dense Retriever saved to: {output_model_path}")

Contrastive-adapted Dense Retriever saved to: ../models/san_diego_legal_bert_contrastive
